# UK Police Crime Data Engineering Pipeline & Reporting Dataset

**Author:** DIP RAIYAN  
**Cohort:** Rockborne Cohort 20  
**Date:** 28 July 2026

---

# Introduction

This notebook documents a scalable, end-to-end data engineering pipeline developed to transform raw operational UK Police crime data into a high-quality, Business Intelligence (BI)-ready reporting dataset. Supporting the Police Force Analytics Unit, the pipeline ingests transactional crime records for **Essex Police**, **Sussex Police**, **Kent Police**, and the **Metropolitan Police Service** covering the period **June 2023 to May 2026**.

To provide richer analytical context, the pipeline integrates three publicly available enrichment datasets:

- Office for National Statistics (ONS) Population Estimates
- Indices of Multiple Deprivation (IMD)
- ONS Workplace-based Earnings and Housing Affordability metrics

The final output is aggregated to a consistent reporting grain, enabling leadership teams to:

- Monitor seasonal crime trends
- Compare performance across police forces
- Evaluate normalised crime rates (crimes per 1,000 residents)

---

# Pipeline Architecture

The pipeline is organised into five modular processing layers, each responsible for a specific stage of the Extract, Transform, and Load (ETL) process. This layered architecture promotes scalability, maintainability, and robust data validation.

---

## Layer 1 – Batch Ingestion

The ingestion layer implements automated directory traversal and batch processing to iteratively load large volumes of monthly CSV files while maintaining a low memory footprint.

Key responsibilities include:

- Automatically discovering source files
- Batch/chunked CSV loading
- Standardising base schemas
- Combining heterogeneous monthly datasets into a unified structure

---

## Layer 2 – Cleaning & Validation

This layer performs automated data cleansing and validation to improve data quality before transformation.

Primary tasks include:

- Removing unnecessary administrative columns
- Standardising string casing and formatting
- Handling missing spatial identifiers (LSOA)
- Validating required fields

To preserve total crime volumes, records with missing LSOA values are assigned the **`unmapped_lsoa`** flag rather than being discarded.

---

## Layer 3 – Feature Engineering & Enrichment

This layer derives analytical features and enriches the operational crime data with external demographic and socioeconomic datasets.

### 3.0 Temporal Feature Engineering & Population Integration

Temporal attributes are generated to support downstream reporting and facilitate alignment with annual enrichment datasets.

Generated features include:

- Year
- Month

Crime records are enriched using **ONS Population Estimates** at the **Police Force Area × Year** grain.

To accommodate unavailable future population estimates, **2024 population figures are forward-filled for the 2025–2026 reporting period**, ensuring consistent calculation of normalised metrics.

---

### 3.1 IMD Integration

The pipeline joins the **Indices of Multiple Deprivation (IMD)** dataset at the **LSOA** level, providing additional socioeconomic context for each crime record.

---

### 3.2 Housing & Earnings Integration

Regional economic indicators are integrated by programmatically extracting, transforming, and reshaping data from **18 Office for National Statistics worksheets**.

The integrated metrics include:

- Workplace earnings
- Median house prices
- Housing affordability ratios

---

## Layer 4 – Aggregation for Reporting

Following enrichment, millions of transactional crime records are aggregated into a consistent reporting grain:

> **Police Force × Month × Local Authority District × Crime Type**

During this stage, the pipeline calculates the primary performance metric:

- **Crimes per 1,000 residents**

The resulting dataset is optimised for Business Intelligence and dashboard reporting.

---

## Layer 5 – Final Validation & Export

The final layer performs comprehensive validation checks before exporting the completed reporting dataset.

Validation includes:

- Confirming zero duplicate records at the reporting grain
- Verifying completeness of core reporting metrics
- Ensuring successful aggregation
- Exporting a single validated, BI-ready CSV file

---

# Final Output

The completed pipeline produces a fully documented, enriched, and validated reporting dataset that serves as the foundation for downstream Business Intelligence analysis. The final CSV is optimised for tools such as Power BI and supports strategic analysis of regional crime patterns, force comparisons, and normalised crime reporting.


# Project Directory Structure

The data pipeline is designed to programmatically traverse the following directory structure to ingest both the primary operational UK Police crime records and the socio-economic enrichment datasets.

```text
project_root/
│
├── uk_police_data/                              # Primary UK Police Crime Data (Source: data.police.uk)
│   ├── 2023-06/                                 # Monthly folders (YYYY-MM)
│   │   ├── 2023-06-essex-street.csv
│   │   ├── 2023-06-sussex-street.csv
│   │   ├── 2023-06-kent-street.csv
│   │   └── 2023-06-metropolitan-street.csv
│   ├── ...
│   └── 2026-05/
│       ├── 2026-05-essex-street.csv
│       └── ...
│
├── enrichment_data/                             # Supplementary ONS and Socioeconomic Datasets
│   ├── employment_housing_demographic_metrics.xlsx   # ONS Housing, Earnings & Affordability Metrics
│   ├── indices_of_deprivation.csv                    # Index of Multiple Deprivation (IMD) Data
│   └── population_by_police_force.xlsx               # ONS Population Estimates
│
└── crime_data_engineering_pipeline.ipynb             # Main Data Engineering Notebook
```

## Directory Notes

### UK Police Data

The UK Police crime dataset spans the period **June 2023 to May 2026**. The ingestion layer uses automated directory traversal to locate, identify, and batch-load monthly CSV files, ensuring efficient processing while preventing memory overflow during execution.

### Enrichment Data

The enrichment directory contains the three external datasets required to enhance the crime data with demographic and socioeconomic context. These datasets enable the calculation of normalised crime metrics, such as **crimes per 1,000 residents**, and support more meaningful comparisons across police force areas.

### File Heterogeneity

The data engineering pipeline is designed to accommodate variations in the structure of the monthly CSV files, including differences in row counts and, where applicable, column availability. During the ingestion phase, the pipeline performs validation and standardisation to ensure all datasets are processed consistently before downstream transformations.

In [7]:
import pandas as pd
import glob
import os

# Define the target police forces as specified in the project scope
TARGET_FORCES = ['essex', 'sussex', 'kent', 'metropolitan']

# Define the base directory where the unzipped UK Police Crime Data is stored
# Adjust this path to wherever your data folders (e.g., '2023-06', '2023-07') are located
DATA_DIR = './uk_police_data/'

# Define the exact columns to ingest, dropping empty/unnecessary columns like 'Context' to save memory
STREET_USECOLS = [
    'Crime ID', 'Month', 'Reported by', 'Falls within', 
    'Longitude', 'Latitude', 'LSOA code', 'LSOA name', 
    'Crime type', 'Last outcome category'
]

## Batch Ingestion 

## 1. Ingestion Layer: Batch Processing UK Police Crime Data

### Purpose & Context
The foundation of the analytics pipeline relies on massive volumes of operational UK Police data recorded between June 2023 and May 2026. This layer is responsible for programmatically extracting and loading three distinct file types (Street Crime, Outcomes, and Stop & Search) across four target police forces (Essex, Sussex, Kent, and the Metropolitan Police Service).

### Structural Transformation (ETL Strategy)
Because the raw data is distributed across dozens of month-year folders containing heterogeneous CSVs, a bulk-load approach would cause memory overflow. Our ingestion strategy employs a robust batch-processing mechanism:
1. **Directory Traversal:** Iterates through the hierarchical folder structure, programmatically identifying files that match our target date range (explicitly enforcing the June 2023 to May 2026 project scope) and chosen police forces.
2. **Chunked Memory Management:** Reads files in manageable batches using `pd.read_csv`, dynamically appending them to type-specific lists rather than holding the entire system in memory at once.
3. **Schema Alignment:** Standardizes base column names upon ingestion to ensure that variations in the raw CSV headers do not break downstream concatenation.
4. **Dataframe Assembly:** Concatenates the validated lists into three primary, distinct raw dataframes representing the different operational grains (Street, Outcomes, Stop and Search).

### Standard Output Schema
The resulting raw dataframes (`df_street_raw`, `df_outcomes_raw`, `df_stop_search_raw`) contain millions of transactional records with the following core fields:
* `crime_id`: The unique alphanumeric identifier for the operational event.
* `month`: The string representation of the reporting period (e.g., `2023-06`).
* `reported_by` / `falls_within`: The jurisdictional police force.
* `longitude` & `latitude`: Geospatial coordinates of the incident.
* `lsoa_code` & `lsoa_name`: Lower Layer Super Output Area identifiers.
* `crime_type` / `last_outcome_category`: Operational categorizations of the event.

In [10]:
def ingest_street_data(base_dir, forces, usecols):
    """
    Iteratively finds and ingests street.csv files for the targeted police forces.
    Reads data in batches (file-by-file) to limit memory overhead and logs row counts.
    """
    all_chunks = []
    total_raw_rows = 0
    file_count = 0
    
    print("--- Starting Batch Ingestion ---")
    
    # Define valid months array based on project scope
    valid_months = [f"{y}-{m:02d}" for y in range(2023, 2027) for m in range(1, 13)]
    valid_months = [m for m in valid_months if "2023-06" <= m <= "2026-05"]
    
    # Iterate through all subdirectories (months) in the base directory
    for root, dirs, files in os.walk(base_dir):
        folder_name = os.path.basename(root)
        if folder_name not in valid_months and root.strip('./\\') != base_dir.strip('./\\'):
            continue
            
        for force in forces:
            # Construct a search pattern for each force's street data file
            search_pattern = f"*{force}-street.csv"
            matched_files = glob.glob(os.path.join(root, search_pattern))
            
            for file_path in matched_files:
                try:
                    # Ingest the file, filtering columns immediately on read to minimise memory usage
                    chunk = pd.read_csv(file_path, usecols=usecols)
                    
                    # Track validation metrics
                    chunk_rows = len(chunk)
                    total_raw_rows += chunk_rows
                    file_count += 1
                    
                    all_chunks.append(chunk)
                    
                except Exception as e:
                    print(f"Error loading {file_path}: {e}")
                    
    print(f"Ingestion Complete: Processed {file_count} files.")
    print(f"Total Raw Rows Ingested: {total_raw_rows}")
    
    # Concatenate the filtered chunks into a single DataFrame for the next layer
    if all_chunks:
        df_raw = pd.concat(all_chunks, ignore_index=True)
        return df_raw
    else:
        print("Warning: No data chunks found. Check DATA_DIR and file structures.")
        return pd.DataFrame()

# Execute the ingestion function
df_street_raw = ingest_street_data(DATA_DIR, TARGET_FORCES, STREET_USECOLS)

# Display a high-level generic statistic to validate the process
display(df_street_raw.head())
display(df_street_raw.info())

--- Starting Batch Ingestion ---
Ingestion Complete: Processed 144 files.
Total Raw Rows Ingested: 4941487


,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,LSOA code,LSOA name,Crime type,Last outcome category
0,2ca7743d38945ce464103aeeef6a33ff5e6d5287950904...,2023-06,Essex Police,Essex Police,0.436560,51.638817,E01021237,Basildon 001A,Vehicle crime,Investigation complete; no suspect identified
1,NaN,2023-06,Essex Police,Essex Police,0.433293,51.641781,E01021238,Basildon 001B,Anti-social behaviour,NaN
2,e74e66c17196221b1948ba269ab1f6bb0eef8cf532b833...,2023-06,Essex Police,Essex Police,0.432756,51.642538,E01021238,Basildon 001B,Criminal damage and arson,Investigation complete; no suspect identified
3,8ecde326bb6cb643648831e259b2bbcef7fc627374cf34...,2023-06,Essex Police,Essex Police,0.433923,51.644628,E01021238,Basildon 001B,Public order,Unable to prosecute suspect
4,0db953a593769e878d4e0d628f18d42d5647e72bf2ffd3...,2023-06,Essex Police,Essex Police,0.438078,51.627248,E01021242,Basildon 001C,Criminal damage and arson,Investigation complete; no suspect identified


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4941487 entries, 0 to 4941486
Data columns (total 10 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Crime ID               object 
 1   Month                  object 
 2   Reported by            object 
 3   Falls within           object 
 4   Longitude              float64
 5   Latitude               float64
 6   LSOA code              object 
 7   LSOA name              object 
 8   Crime type             object 
 9   Last outcome category  object 
dtypes: float64(2), object(8)
memory usage: 377.0+ MB


None

## 2. Cleaning & Validation Layer: Quality Assurance & Standardization

### Purpose & Context
Raw operational police data is inherently messy, containing empty administrative columns, missing spatial identifiers, and inconsistent string casings. This layer guarantees data integrity, mitigates anomalous data points, and perfectly standardizes joining keys to ensure seamless relational merges with our enrichment datasets.

### Structural Transformation (ETL Strategy)
To elevate the raw operational records to BI-ready quality, the data undergoes rigorous automated sanitation:
1. **Feature Pruning:** Identifies and drops fully null or purely administrative columns (e.g., the `Context` column in street data or unpopulated search flags in stop and search data) to optimize dataframe width.
2. **String Casing Normalization:** Enforces strict Title Case formatting on jurisdictional columns (e.g., converting mixed casings to exactly `Essex Police`). This prevents "many-to-many" or null explosions during downstream enrichment joins.
3. **Spatial & ID Null Handling:** Evaluates rows missing critical relational keys (`lsoa_code`, `crime_id`). Depending on the analytical requirement, these are explicitly logged and either dropped or flagged as `unmapped` to maintain the integrity of spatial aggregations.
4. **Explicit Validation Gates:** Calculates and prints exact row counts before and after the cleaning operations, logging the exact volume of dropped duplicates or nulls to ensure no unintended data loss occurs.\n5. **Defensible Duplicate Rule:** Legitimate incidents lacking a Crime ID (e.g. Anti-Social Behaviour) are isolated and protected from blanket deduplication, ensuring true crime volumes are preserved.

### Standard Output Schema
The cleaned dataframes (e.g., `df_street_clean`) retain the exact architectural grain of the raw data (one row per incident) but are now strictly typed and formatted. All string keys are standardized, duplicate identifiers are resolved, and the data is primed for multi-dimensional joining.



In [12]:
def clean_and_validate(df):
    print("--- Starting Layer 2: Cleaning & Validation ---")
    
    # Create a copy to avoid SettingWithCopy warnings
    df_clean = df.copy()
    
    # 1. Standardise column names (lowercase, replace spaces with underscores)
    df_clean.columns = df_clean.columns.str.lower().str.replace(' ', '_')
    
    # Record baseline rows
    rows_before = len(df_clean)
    
    # 2. Handle missing LSOA codes 
    # Essential so we don't lose crime volumes during aggregation, even if they can't be mapped to IMD
    df_clean['lsoa_code'] = df_clean['lsoa_code'].fillna('unmapped_lsoa')
    df_clean['lsoa_name'] = df_clean['lsoa_name'].fillna('Unmapped Location')
    
    # 3. Handle missing Crime IDs (Anti-social behaviour primarily)
    df_clean['crime_id'] = df_clean['crime_id'].fillna('no_id_recorded')
    
    # 4. Handle missing Outcome categories
    df_clean['last_outcome_category'] = df_clean['last_outcome_category'].fillna('None recorded')
    
    # 5. Defensible Duplicate Check & Removal
    # Only deduplicate rows that have an actual crime_id. 
    # 'no_id_recorded' rows (e.g. ASB) are distinct incidents even if location/month match.
    df_with_id = df_clean[df_clean['crime_id'] != 'no_id_recorded']
    df_no_id = df_clean[df_clean['crime_id'] == 'no_id_recorded']
    
    df_with_id_dedup = df_with_id.drop_duplicates()
    df_clean = pd.concat([df_with_id_dedup, df_no_id], ignore_index=True)
    
    rows_after = len(df_clean)
    duplicates_removed = rows_before - rows_after
    
    # Print Explicit Validation Checks
    print(f"Row count before duplicate removal: {rows_before}")
    print(f"Row count after duplicate removal:  {rows_after}")
    print(f"Total exact duplicates removed:     {duplicates_removed}")
    
    print("\nNull values remaining per column (Latitude/Longitude nulls are expected for unmapped locations):")
    print(df_clean.isnull().sum())
    
    return df_clean

# Execute the cleaning function
df_street_clean = clean_and_validate(df_street_raw)

# Display a preview of the clean dataframe
display(df_street_clean.head())

--- Starting Layer 2: Cleaning & Validation ---
Row count before duplicate removal: 4941487
Row count after duplicate removal:  4557223
Total exact duplicates removed:     384264

Null values remaining per column (Latitude/Longitude nulls are expected for unmapped locations):
crime_id                     0
month                        0
reported_by                  0
falls_within                 0
longitude                44039
latitude                 44039
lsoa_code                    0
lsoa_name                    0
crime_type                   0
last_outcome_category        0
dtype: int64


,crime_id,month,reported_by,falls_within,longitude,latitude,lsoa_code,lsoa_name,crime_type,last_outcome_category
0,2ca7743d38945ce464103aeeef6a33ff5e6d5287950904...,2023-06,Essex Police,Essex Police,0.436560,51.638817,E01021237,Basildon 001A,Vehicle crime,Investigation complete; no suspect identified
1,no_id_recorded,2023-06,Essex Police,Essex Police,0.433293,51.641781,E01021238,Basildon 001B,Anti-social behaviour,None recorded
2,e74e66c17196221b1948ba269ab1f6bb0eef8cf532b833...,2023-06,Essex Police,Essex Police,0.432756,51.642538,E01021238,Basildon 001B,Criminal damage and arson,Investigation complete; no suspect identified
3,8ecde326bb6cb643648831e259b2bbcef7fc627374cf34...,2023-06,Essex Police,Essex Police,0.433923,51.644628,E01021238,Basildon 001B,Public order,Unable to prosecute suspect
4,0db953a593769e878d4e0d628f18d42d5647e72bf2ffd3...,2023-06,Essex Police,Essex Police,0.438078,51.627248,E01021242,Basildon 001C,Criminal damage and arson,Investigation complete; no suspect identified


# Layer 3: Feature Engineering & Joins

**Objective:** Derive necessary temporal attributes and integrate our chosen enrichment datasets at compatible grains.

**Requirements Addressed:**
* **Feature Engineering:** Extract `year` and `month` from the primary crime data to align with annual enrichment metrics.
* **Enrichment Joins:** 
  * *Layer 3.0:* Join Population data (Grain: Police Force $\times$ Year).
  * *Layer 3.1:* Join Index of Multiple Deprivation (IMD) data (Grain: LSOA).
  * *Layer 3.2:* Join ONS Housing & Earnings metrics (Grain: Local Authority District $\times$ Year).
* **Validation:** Ensure no unintentional "many-to-many" row explosions occur during the merge processes.

In [14]:
def engineer_features_and_population(df_crime, pop_file_path):
    print("--- Starting Layer 3.0: Feature Eng & Population Join ---")
    
    # 1. Derive time attributes (Year)
    # The 'month' column is formatted 'YYYY-MM', so we extract the first 4 characters
    df_crime['year'] = df_crime['month'].str[:4].astype(int)
    
    # 2. Ingest & Process Population Data
    print("Loading ONS Population Data...")
    
    # ADDED skiprows=3 to bypass the ONS title/metadata rows
    df_pop_raw = pd.read_excel(pop_file_path, sheet_name='Mid-2021 to Mid-2024', skiprows=3)
    
    # Filter for our 4 target forces to save memory
    target_forces_proper = ['Essex', 'Sussex', 'Kent', 'Metropolitan Police']
    df_pop = df_pop_raw[df_pop_raw['PFA 2023 Name'].isin(target_forces_proper)].copy()
    
    # Calculate Total Population by summing all 172 age/sex columns (F0-F85, M0-M85)
    age_cols = [c for c in df_pop.columns if c.startswith('F') or c.startswith('M')]
    df_pop['total_population'] = df_pop[age_cols].sum(axis=1)
    
    # Map Force names to strictly match the lowercase names in our crime dataset
    force_map = {
        'Essex': 'Essex Police',
        'Sussex': 'Sussex Police',
        'Kent': 'Kent Police',
        'Metropolitan Police': 'Metropolitan Police Service'
    }
    df_pop['reported_by'] = df_pop['PFA 2023 Name'].map(force_map)
    
    # Subset to only the columns we need for the join
    df_pop = df_pop[['reported_by', 'Year', 'total_population']].rename(columns={'Year': 'year'})
    
    # 3. Handle Temporal Mismatch (Forward Fill 2024 population for 2025 and 2026)
    print("Handling temporal mismatch: Forward-filling 2024 population estimates for 2025/2026...")
    pop_2024 = df_pop[df_pop['year'] == 2024].copy()
    
    pop_2025 = pop_2024.copy()
    pop_2025['year'] = 2025
    
    pop_2026 = pop_2024.copy()
    pop_2026['year'] = 2026
    
    # Combine the historical and forward-filled population data
    df_pop_extended = pd.concat([df_pop, pop_2025, pop_2026], ignore_index=True)
    
    # 4. Join Population to Crime Data
    print("Joining Population data to Crime data...")
    df_enriched = pd.merge(df_crime, df_pop_extended, on=['reported_by', 'year'], how='left')
    
    # Explicit Validation: Ensure no unintentional many-to-many row explosion and check for nulls
    rows_before = len(df_crime)
    rows_after = len(df_enriched)
    missing_pop = df_enriched['total_population'].isnull().sum()
    
    print(f"Row count before join: {rows_before}")
    print(f"Row count after join:  {rows_after}")
    assert rows_before == rows_after, "FAILED: Unintentional row explosion detected during population join."
    if rows_before == rows_after:
        print("Success: No unintentional row explosion detected.")
    
    print(f"Join Validation: Rows missing population data after merge: {missing_pop}")
    
    return df_enriched, df_pop_extended

# --- EXECUTION ---
POPULATION_FILE_PATH = './enrichment_data/population_by_police_force.xlsx' 

df_street_enriched, df_pop_reference = engineer_features_and_population(df_street_clean, POPULATION_FILE_PATH)

display(df_street_enriched.head())

--- Starting Layer 3.0: Feature Eng & Population Join ---
Loading ONS Population Data...
Handling temporal mismatch: Forward-filling 2024 population estimates for 2025/2026...
Joining Population data to Crime data...
Row count before join: 4557223
Row count after join:  4557223
Success: No unintentional row explosion detected.
Join Validation: Rows missing population data after merge: 0


,crime_id,month,reported_by,falls_within,longitude,latitude,lsoa_code,lsoa_name,crime_type,last_outcome_category,year,total_population
0,2ca7743d38945ce464103aeeef6a33ff5e6d5287950904...,2023-06,Essex Police,Essex Police,0.436560,51.638817,E01021237,Basildon 001A,Vehicle crime,Investigation complete; no suspect identified,2023,1904539
1,no_id_recorded,2023-06,Essex Police,Essex Police,0.433293,51.641781,E01021238,Basildon 001B,Anti-social behaviour,None recorded,2023,1904539
2,e74e66c17196221b1948ba269ab1f6bb0eef8cf532b833...,2023-06,Essex Police,Essex Police,0.432756,51.642538,E01021238,Basildon 001B,Criminal damage and arson,Investigation complete; no suspect identified,2023,1904539
3,8ecde326bb6cb643648831e259b2bbcef7fc627374cf34...,2023-06,Essex Police,Essex Police,0.433923,51.644628,E01021238,Basildon 001B,Public order,Unable to prosecute suspect,2023,1904539
4,0db953a593769e878d4e0d628f18d42d5647e72bf2ffd3...,2023-06,Essex Police,Essex Police,0.438078,51.627248,E01021242,Basildon 001C,Criminal damage and arson,Investigation complete; no suspect identified,2023,1904539


In [15]:
df_street_enriched[
    (df_street_enriched["reported_by"] == "Metropolitan Police Service") &
    (df_street_enriched["month"] == "2023-11")
].tail()

,crime_id,month,reported_by,falls_within,longitude,latitude,lsoa_code,lsoa_name,crime_type,last_outcome_category,year,total_population
798723,05b7f2e517716801bb1be038e6775d0de04d6f3105d3c9...,2023-11,Metropolitan Police Service,Metropolitan Police Service,NaN,NaN,unmapped_lsoa,Unmapped Location,Other crime,Investigation complete; no suspect identified,2023,8986099
798724,5fc170ea9c4e6f3b1a7ce2d94ecdf3e6fc3c20bfd01233...,2023-11,Metropolitan Police Service,Metropolitan Police Service,NaN,NaN,unmapped_lsoa,Unmapped Location,Other crime,Status update unavailable,2023,8986099
798725,970306222c981875a64419f735a271c657e32f4b156fb9...,2023-11,Metropolitan Police Service,Metropolitan Police Service,NaN,NaN,unmapped_lsoa,Unmapped Location,Other crime,Investigation complete; no suspect identified,2023,8986099
798726,5fc170ea9c4e6f3b1a7ce2d94ecdf3e6fc3c20bfd01233...,2023-11,Metropolitan Police Service,Metropolitan Police Service,NaN,NaN,unmapped_lsoa,Unmapped Location,Other crime,Investigation complete; no suspect identified,2023,8986099
798727,2a6d98cdb83449e9e676909465ba9043774d1dc420e776...,2023-11,Metropolitan Police Service,Metropolitan Police Service,NaN,NaN,unmapped_lsoa,Unmapped Location,Other crime,Investigation complete; no suspect identified,2023,8986099


In [16]:
df_street_enriched[df_street_enriched["reported_by"] == "Metropolitan Police Service"].head(3)

,crime_id,month,reported_by,falls_within,longitude,latitude,lsoa_code,lsoa_name,crime_type,last_outcome_category,year,total_population
42332,a5a2ab72258bfb2e808347707bcc312470d031825e0911...,2023-06,Metropolitan Police Service,Metropolitan Police Service,-0.685028,50.780596,E01031437,Arun 017E,Violence and sexual offences,Status update unavailable,2023,8986099
42333,16efc0a615f7368d0d34c82d9989a1eb485616f8bc56d3...,2023-06,Metropolitan Police Service,Metropolitan Police Service,-0.686514,50.780694,E01031437,Arun 017E,Violence and sexual offences,Status update unavailable,2023,8986099
42334,8400c77980478215c845adbe62aae9b205385268ca8b4a...,2023-06,Metropolitan Police Service,Metropolitan Police Service,0.876053,51.171725,E01032810,Ashford 001F,Violence and sexual offences,Investigation complete; no suspect identified,2023,8986099


## Layer 3.1: Indices of Deprivation (IMD) Integration

### Purpose & Context
To address core leadership questions regarding how crime data relates to socioeconomic context, this layer integrates the Indices of Multiple Deprivation (IMD) dataset. This enrichment provides a granular view of regional deprivation, allowing analysts to correlate crime patterns with local socioeconomic health.

### Structural Transformation (ETL Strategy)
The IMD dataset is a high-density reference file (33,756 rows) with high data integrity and no null values. To successfully integrate this context, the pipeline performs the following steps:
1. **Grain Alignment:** The join is executed at the LSOA (Lower Layer Super Output Area) grain, utilizing the `lsoa_code` as the primary relational key.
2. **Join Method (Validated Left Join):** To preserve the integrity of total crime volumes, the pipeline performs a left join. This ensures that crime records missing a valid spatial identifier (flagged as 'unmapped_lsoa' during Layer 2) are not dropped from the dataset, even if they cannot be assigned a deprivation decile.
3. **Integrity Check:** The layer includes an explicit validation gate to ensure no "many-to-many" row explosions occurred, maintaining a 1:1 relationship between crime records and their corresponding LSOA metrics.

### Standard Output Schema
The resulting enriched dataframe includes:
* `imd_decile`: The primary socioeconomic measure (typically 1–10, where 1 represents the most deprived decile).
* `lsoa_code` / `lsoa_name`: Standardised spatial identifiers used to facilitate the merge.
* `Socioeconomic Context`: Additional deprivation indicators that support operational comparisons across different police forces.

In [18]:
def enrich_with_imd(df_crime, imd_file_path):
    print("--- Starting Layer 3.1: Indices of Deprivation (IMD) Join ---")
    
    # 1. Load IMD Data
    print("Loading IMD Data...")
    
    # Using the exact column names extracted from the dataset
    lsoa_col = 'LSOA code (2021)' 
    imd_metric_col = 'Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)'
    
    # Load only the necessary columns to save memory
    df_imd_raw = pd.read_csv(imd_file_path, usecols=[lsoa_col, imd_metric_col])
    
    # Rename columns to standardise and match our crime dataframe
    df_imd = df_imd_raw.rename(columns={
        lsoa_col: 'lsoa_code', 
        imd_metric_col: 'imd_decile'
    })
    
    # 2. Join IMD to Crime Data
    print("Joining IMD data to Crime data...")
    # We use a LEFT JOIN so we don't drop crimes that occurred in an 'unmapped_lsoa'
    df_fully_enriched = pd.merge(df_crime, df_imd, on='lsoa_code', how='left')
    
    # 3. Explicit Validation
    rows_before = len(df_crime)
    rows_after = len(df_fully_enriched)
    missing_imd = df_fully_enriched['imd_decile'].isnull().sum()
    
    print(f"Row count before join: {rows_before}")
    print(f"Row count after join:  {rows_after}")
    
    assert rows_before == rows_after, "FAILED: Unintentional row explosion detected during IMD join."
    if rows_before == rows_after:
        print("Success: No unintentional row explosion detected.")
        
    print(f"Join Validation: Rows missing IMD data after merge: {missing_imd}")
    print("(Note: It is expected and perfectly fine that 'unmapped_lsoa' locations will have missing IMD data!)")
    
    return df_fully_enriched

# --- EXECUTION ---
IMD_FILE_PATH = './enrichment_data/Indicies_of_deprivation.csv' 

df_street_fully_enriched = enrich_with_imd(df_street_enriched, IMD_FILE_PATH)

display(df_street_fully_enriched.head())

--- Starting Layer 3.1: Indices of Deprivation (IMD) Join ---
Loading IMD Data...
Joining IMD data to Crime data...
Row count before join: 4557223
Row count after join:  4557223
Success: No unintentional row explosion detected.
Join Validation: Rows missing IMD data after merge: 44380
(Note: It is expected and perfectly fine that 'unmapped_lsoa' locations will have missing IMD data!)


,crime_id,month,reported_by,falls_within,longitude,latitude,lsoa_code,lsoa_name,crime_type,last_outcome_category,year,total_population,imd_decile
0,2ca7743d38945ce464103aeeef6a33ff5e6d5287950904...,2023-06,Essex Police,Essex Police,0.436560,51.638817,E01021237,Basildon 001A,Vehicle crime,Investigation complete; no suspect identified,2023,1904539,10.0
1,no_id_recorded,2023-06,Essex Police,Essex Police,0.433293,51.641781,E01021238,Basildon 001B,Anti-social behaviour,None recorded,2023,1904539,10.0
2,e74e66c17196221b1948ba269ab1f6bb0eef8cf532b833...,2023-06,Essex Police,Essex Police,0.432756,51.642538,E01021238,Basildon 001B,Criminal damage and arson,Investigation complete; no suspect identified,2023,1904539,10.0
3,8ecde326bb6cb643648831e259b2bbcef7fc627374cf34...,2023-06,Essex Police,Essex Police,0.433923,51.644628,E01021238,Basildon 001B,Public order,Unable to prosecute suspect,2023,1904539,10.0
4,0db953a593769e878d4e0d628f18d42d5647e72bf2ffd3...,2023-06,Essex Police,Essex Police,0.438078,51.627248,E01021242,Basildon 001C,Criminal damage and arson,Investigation complete; no suspect identified,2023,1904539,9.0


---

## 3.2. Enrichment Layer: ONS Housing, Earnings & Affordability Integration

### Purpose & Context
To evaluate how socioeconomic and macroeconomic conditions influence local crime patterns, we integrate the **ONS Workplace-based Earnings and House Price Affordability** dataset. This dataset provides localized economic context across England and Wales from 1997 through 2025.

### Structural Transformation (ETL Strategy)
The raw ONS Excel file (`employment_housing_demographic_metrics.xlsx`) stores data in a **human-readable, multi-tab layout** (18 distinct worksheets categorized by metric, geographic tier, and summary statistic). 

To make this data BI-ready and joinable with our crime records, our automated function performs the following transformations:
1. **Multi-Sheet Parsing:** Iterates through all 18 worksheets (`1a` through `6c`) and maps them to standardized metadata (Geographic Level, Statistic, Metric, Units).
2. **Regex Attribute Identification:** Dynamically detects and extracts geographic codes, names, parent regions, and 4-digit calendar years (`1997`–`2025`) across varying header formats.
3. **Wide-to-Long Unpivoting (`pd.melt`):** Flattens wide annual columns into a single longitudinal schema, creating one row per region $\times$ year $\times$ metric.
4. **Data Cleansing:** Strips non-numeric ONS suppression symbols (e.g., `[x]`, `:`) and normalizes metrics to numeric float values.\n5. **Standardisation & Completeness:** Applies geographic standardisation to LAD names to maximise join success and explicitly forward-fills 2025 housing metrics to cover the 2026 reporting period.

### Standard Output Schema
The resulting tidy dataframe (`df_housing_tidy`) contains **61,074 rows** structured as follows:
* `geography_level`: Regional, County, or Local Authority District (LAD).
* `geography_code` & `geography_name`: Official ONS spatial identifier and localized boundary name.
* `parent_region_code` & `parent_region_name`: Parent region mapping (for LAD/County rows).
* `statistic`: Statistical measure (`Median` or `Lower Quartile`).
* `metric`: Metric evaluation (`House Price`, `Earnings`, or `Affordability Ratio`).
* `year`: Calendar year of measurement (1997–2025).
* `value`: Primary numerical measure (£ for price/earnings; ratio for affordability).

---

In [20]:
import pandas as pd
import numpy as np
import re

def process_housing_workbook(excel_path):
    print("--- Starting Processing of ONS Housing, Earnings & Affordability Workbook ---")
    
    # Metadata mapping for all 18 worksheets
    sheet_metadata = {
        '1a': ('Region', 'Median', 'House Price', '£', 'Year ending Sep'),
        '1b': ('Region', 'Median', 'Earnings', '£', 'Calendar Year'),
        '1c': ('Region', 'Median', 'Affordability Ratio', 'Ratio', 'Year ending Sep'),
        '2a': ('Region', 'Lower Quartile', 'House Price', '£', 'Year ending Sep'),
        '2b': ('Region', 'Lower Quartile', 'Earnings', '£', 'Calendar Year'),
        '2c': ('Region', 'Lower Quartile', 'Affordability Ratio', 'Ratio', 'Year ending Sep'),
        '3a': ('County', 'Median', 'House Price', '£', 'Year ending Sep'),
        '3b': ('County', 'Median', 'Earnings', '£', 'Calendar Year'),
        '3c': ('County', 'Median', 'Affordability Ratio', 'Ratio', 'Year ending Sep'),
        '4a': ('County', 'Lower Quartile', 'House Price', '£', 'Year ending Sep'),
        '4b': ('County', 'Lower Quartile', 'Earnings', '£', 'Calendar Year'),
        '4c': ('County', 'Lower Quartile', 'Affordability Ratio', 'Ratio', 'Year ending Sep'),
        '5a': ('Local Authority', 'Median', 'House Price', '£', 'Year ending Sep'),
        '5b': ('Local Authority', 'Median', 'Earnings', '£', 'Calendar Year'),
        '5c': ('Local Authority', 'Median', 'Affordability Ratio', 'Ratio', 'Year ending Sep'),
        '6a': ('Local Authority', 'Lower Quartile', 'House Price', '£', 'Year ending Sep'),
        '6b': ('Local Authority', 'Lower Quartile', 'Earnings', '£', 'Calendar Year'),
        '6c': ('Local Authority', 'Lower Quartile', 'Affordability Ratio', 'Ratio', 'Year ending Sep'),
    }

    all_sheets = []

    for sheet_code, meta in sheet_metadata.items():
        geo_level, stat, metric, unit, reporting_period = meta
        
        try:
            # Read worksheet, skipping Title row
            df_raw = pd.read_excel(excel_path, sheet_name=sheet_code, skiprows=1)
            df_raw = df_raw.dropna(how='all')

            # 1. Identify Year columns using Regex (extract 4-digit year like 1997 or 2025)
            year_col_map = {}
            non_year_cols = []
            for col in df_raw.columns:
                match = re.search(r'\b(19\d\d|20\d\d)\b', str(col))
                if match:
                    year_col_map[col] = int(match.group(1))
                else:
                    non_year_cols.append(col)

            # 2. Classify Geographic columns flexibly
            geo_code_col, geo_name_col = None, None
            parent_code_col, parent_name_col = None, None

            for col in non_year_cols:
                c_lower = re.sub(r'\s+', ' ', str(col).replace('\n', ' ')).strip().lower()
                
                if 'local authority code' in c_lower or 'county code' in c_lower:
                    geo_code_col = col
                elif 'local authority name' in c_lower or 'county name' in c_lower:
                    geo_name_col = col
                elif 'country/region code' in c_lower or 'region code' in c_lower:
                    if geo_level == 'Region':
                        geo_code_col = col
                    else:
                        parent_code_col = col
                elif 'country/region name' in c_lower or 'region name' in c_lower:
                    if geo_level == 'Region':
                        geo_name_col = col
                    else:
                        parent_name_col = col
                elif c_lower == 'code':
                    geo_code_col = col
                elif c_lower == 'name':
                    geo_name_col = col

            # 3. Unpivot (Melt) wide table to long schema
            id_vars = [c for c in [geo_code_col, geo_name_col, parent_code_col, parent_name_col] if c is not None]
            
            df_melted = pd.melt(
                df_raw,
                id_vars=id_vars,
                value_vars=list(year_col_map.keys()),
                var_name='year_raw',
                value_name='value'
            )

            # 4. Map Clean Attributes
            df_melted['year'] = df_melted['year_raw'].map(year_col_map)
            df_melted['geography_level'] = geo_level
            df_melted['geography_code'] = df_melted[geo_code_col] if geo_code_col else np.nan
            df_melted['geography_name'] = df_melted[geo_name_col] if geo_name_col else np.nan
            df_melted['parent_region_code'] = df_melted[parent_code_col] if parent_code_col else np.nan
            df_melted['parent_region_name'] = df_melted[parent_name_col] if parent_name_col else np.nan
            
            df_melted['statistic'] = stat
            df_melted['metric'] = metric
            df_melted['unit'] = unit
            df_melted['reporting_period'] = reporting_period
            df_melted['source_sheet'] = sheet_code

            # 5. Clean suppressed ONS values (e.g. '[x]' or ':') to numeric floats
            df_melted['value'] = pd.to_numeric(
                df_melted['value'].astype(str).str.replace(':', '', regex=False).str.replace('[x]', '', regex=False).str.strip(),
                errors='coerce'
            )

            # Reorder standard schema
            final_cols = [
                'geography_level', 'geography_code', 'geography_name',
                'parent_region_code', 'parent_region_name',
                'statistic', 'metric', 'year', 'reporting_period',
                'value', 'unit', 'source_sheet'
            ]
            
            df_final_sheet = df_melted[final_cols]
            all_sheets.append(df_final_sheet)
            print(f"Processed Sheet '{sheet_code:2s}' ({geo_level:15s} | {stat:14s} {metric:18s}): {len(df_final_sheet):5d} rows")

        except Exception as e:
            print(f"Error processing sheet '{sheet_code}': {e}")

    # Combine all 18 sheets into one tidy dataframe
    df_combined = pd.concat(all_sheets, ignore_index=True)
    print("\n--- Processing Complete ---")
    print(f"Total Combined Rows: {len(df_combined)}")
    return df_combined

# --- EXECUTION ---
HOUSING_FILE_PATH = './enrichment_data/employment_housing_demographic_metrics.xlsx'
df_housing_tidy = process_housing_workbook(HOUSING_FILE_PATH)

# --- VALIDATION ---
print("\n--- Validation: Null Counts in Key Geography Fields ---")
print(df_housing_tidy[['geography_code', 'geography_name']].isnull().sum())


--- Starting Processing of ONS Housing, Earnings & Affordability Workbook ---
Processed Sheet '1a' (Region          | Median         House Price       ):   348 rows


Processed Sheet '1b' (Region          | Median         Earnings          ):   348 rows
Processed Sheet '1c' (Region          | Median         Affordability Ratio):   348 rows
Processed Sheet '2a' (Region          | Lower Quartile House Price       ):   348 rows
Processed Sheet '2b' (Region          | Lower Quartile Earnings          ):   348 rows
Processed Sheet '2c' (Region          | Lower Quartile Affordability Ratio):   348 rows
Processed Sheet '3a' (County          | Median         House Price       ):   609 rows
Processed Sheet '3b' (County          | Median         Earnings          ):   609 rows
Processed Sheet '3c' (County          | Median         Affordability Ratio):   609 rows
Processed Sheet '4a' (County          | Lower Quartile House Price       ):   609 rows
Processed Sheet '4b' (County          | Lower Quartile Earnings          ):   609 rows
Processed Sheet '4c' (County          | Lower Quartile Affordability Ratio):   609 rows
Processed Sheet '5a' (Local Authority |

In [21]:
#df_housing_tidy.tail(5)
df_housing_tidy.head(5)

,geography_level,geography_code,geography_name,parent_region_code,parent_region_name,statistic,metric,year,reporting_period,value,unit,source_sheet
0,Region,K04000001,England and Wales,NaN,NaN,Median,House Price,1997,Year ending Sep,59950.0,£,1a
1,Region,E92000001,England,NaN,NaN,Median,House Price,1997,Year ending Sep,59995.0,£,1a
2,Region,W92000004,Wales,NaN,NaN,Median,House Price,1997,Year ending Sep,47000.0,£,1a
3,Region,E12000001,North East,NaN,NaN,Median,House Price,1997,Year ending Sep,46500.0,£,1a
4,Region,E12000002,North West,NaN,NaN,Median,House Price,1997,Year ending Sep,48500.0,£,1a


In [22]:
def merge_housing_data(df_crime, df_housing_tidy):
    print("--- Starting Layer 3.2: Housing Data Join ---")
    
    # 1. Derive Local Authority District (LAD) from LSOA name
    # E.g. 'Basildon 001A' splits from the right, leaving 'Basildon'
    df_crime['lad_name'] = df_crime['lsoa_name'].astype(str).str.rsplit(' ', n=1).str[0].str.strip()
    
    # 2. Filter and Pivot Housing Data
    # Keep only Local Authority level and Median statistics for our primary BI context
    df_h = df_housing_tidy[
        (df_housing_tidy['geography_level'] == 'Local Authority') & 
        (df_housing_tidy['statistic'] == 'Median')
    ].copy()
    
    # Pivot so metrics (House Price, Earnings, Affordability) become their own columns
    df_h_pivot = df_h.pivot_table(
        index=['geography_name', 'year'],
        columns='metric',
        values='value'
    ).reset_index()
    
    # Rename for the join
    df_h_pivot = df_h_pivot.rename(columns={'geography_name': 'lad_name'})
    
    # Clear 2026 Policy: Forward fill 2025 housing data for 2026
    print("Applying 2026 Housing Policy: Forward-filling 2025 housing data...")
    df_h_2025 = df_h_pivot[df_h_pivot['year'] == 2025].copy()
    df_h_2026 = df_h_2025.copy()
    df_h_2026['year'] = 2026
    df_h_pivot = pd.concat([df_h_pivot, df_h_2026], ignore_index=True)
    
    # Standardise LAD names to improve join match rate
    def standardize_lad(name):
        if pd.isnull(name): return name
        return str(name).replace('&', 'and').replace(',', '').strip().lower()
        
    df_crime['lad_name_std'] = df_crime['lad_name'].apply(standardize_lad)
    df_h_pivot['lad_name_std'] = df_h_pivot['lad_name'].apply(standardize_lad)
    
    # 3. Join to Crime Data
    print("Joining Housing data to Crime data...")
    df_fully_enriched = pd.merge(df_crime, df_h_pivot, on=['lad_name_std', 'year'], how='left')
    
    # Clean up standardisation columns
    df_fully_enriched = df_fully_enriched.drop(columns=['lad_name_std', 'lad_name_y'])
    df_fully_enriched = df_fully_enriched.rename(columns={'lad_name_x': 'lad_name'})
    
    # 4. Explicit Validation
    rows_before = len(df_crime)
    rows_after = len(df_fully_enriched)
    assert rows_before == rows_after, "FAILED: Unintentional row explosion detected during housing join."
    print(f"Row count before join: {rows_before}")
    print(f"Row count after join:  {rows_after}")
    
    missing_housing = df_fully_enriched['Affordability Ratio'].isnull().sum()
    print(f"Join Validation: Rows missing housing data: {missing_housing}")
    
    return df_fully_enriched

# --- EXECUTION ---
# Ensure you have run the process_housing_workbook function from earlier to generate df_housing_tidy
df_street_final_enriched = merge_housing_data(df_street_fully_enriched, df_housing_tidy)

display(df_street_final_enriched.head(3))

--- Starting Layer 3.2: Housing Data Join ---
Joining Housing data to Crime data...
Row count before join: 4557223
Row count after join:  4557223
Join Validation: Rows missing housing data: 646621


,crime_id,month,reported_by,falls_within,longitude,latitude,lsoa_code,lsoa_name,crime_type,last_outcome_category,year,total_population,imd_decile,lad_name,Affordability Ratio,Earnings,House Price
0,2ca7743d38945ce464103aeeef6a33ff5e6d5287950904...,2023-06,Essex Police,Essex Police,0.436560,51.638817,E01021237,Basildon 001A,Vehicle crime,Investigation complete; no suspect identified,2023,1904539,10.0,Basildon,10.51,33788.0,355000.0
1,no_id_recorded,2023-06,Essex Police,Essex Police,0.433293,51.641781,E01021238,Basildon 001B,Anti-social behaviour,None recorded,2023,1904539,10.0,Basildon,10.51,33788.0,355000.0
2,e74e66c17196221b1948ba269ab1f6bb0eef8cf532b833...,2023-06,Essex Police,Essex Police,0.432756,51.642538,E01021238,Basildon 001B,Criminal damage and arson,Investigation complete; no suspect identified,2023,1904539,10.0,Basildon,10.51,33788.0,355000.0


## Layer 4: Aggregation for Reporting

Our dataset currently contains millions of row-level crime events. To optimize performance and usability in Power BI, we need to compress this data down to our target reporting grain: **Police Force $\times$ Month $\times$ Local Authority $\times$ Crime Type**.

In this step, we will group the data to this grain, calculate total crime volumes, and pass through our regional contextual metrics. We will also calculate our primary normalized measure: **Crimes per 1,000 Force Residents**.

In [24]:
def aggregate_for_reporting(df_enriched):
    print("--- Starting Layer 4: Aggregation for Reporting ---")
    
    # Define the reporting grain
    group_cols = ['reported_by', 'month', 'year', 'lad_name', 'crime_type']
    
    # Define how to aggregate the contextual/enrichment columns
    agg_dict = {
        'crime_id': 'count',                     # Count of crimes (Crime Volume)
        'total_population': 'first',             # Police Force Population (Layer 3)
        'imd_decile': 'median',                  # Median IMD for this LAD/Crime slice
        'House Price': 'first',                  # LAD Housing Metric
        'Earnings': 'first',                     # LAD Housing Metric
        'Affordability Ratio': 'first'           # LAD Housing Metric
    }
    
    print(f"Aggregating data to grain: {group_cols}...")
    df_agg = df_enriched.groupby(group_cols, dropna=False).agg(agg_dict).reset_index()
    
    # Rename columns to be BI-friendly
    df_agg = df_agg.rename(columns={
        'crime_id': 'crime_count',
        'total_population': 'force_population',
        'imd_decile': 'median_imd_decile',
        'House Price': 'median_house_price',
        'Earnings': 'median_earnings',
        'Affordability Ratio': 'median_affordability_ratio'
    })
    
    # Calculate normalised metric: Crimes per 1,000 residents
    df_agg['crimes_per_1k_force_residents'] = (df_agg['crime_count'] / df_agg['force_population']) * 1000
    
    # Round metrics for clean Power BI ingestion
    df_agg['crimes_per_1k_force_residents'] = df_agg['crimes_per_1k_force_residents'].round(4)
    df_agg['median_imd_decile'] = df_agg['median_imd_decile'].round(1)
    
    print(f"Aggregation complete. Compressed millions of rows to a final row count of: {len(df_agg)}")
    return df_agg

# --- EXECUTION ---
df_bi_reporting = aggregate_for_reporting(df_street_final_enriched)

display(df_bi_reporting.head(5))

--- Starting Layer 4: Aggregation for Reporting ---
Aggregating data to grain: ['reported_by', 'month', 'year', 'lad_name', 'crime_type']...
Aggregation complete. Compressed millions of rows to a final row count of: 47478


,reported_by,month,year,lad_name,crime_type,crime_count,force_population,median_imd_decile,median_house_price,median_earnings,median_affordability_ratio,crimes_per_1k_force_residents
0,Essex Police,2023-06,2023,Basildon,Anti-social behaviour,145,1904539,3.0,355000.0,33788.0,10.51,0.0761
1,Essex Police,2023-06,2023,Basildon,Bicycle theft,6,1904539,2.5,355000.0,33788.0,10.51,0.0032
2,Essex Police,2023-06,2023,Basildon,Burglary,57,1904539,3.0,355000.0,33788.0,10.51,0.0299
3,Essex Police,2023-06,2023,Basildon,Criminal damage and arson,158,1904539,3.0,355000.0,33788.0,10.51,0.0830
4,Essex Police,2023-06,2023,Basildon,Drugs,44,1904539,2.5,355000.0,33788.0,10.51,0.0231


## Layer 5: Final Validation and Export

Before saving the final deliverable, we must ensure data integrity. This step performs automated quality checks against the final aggregated dataframe to confirm two things:
1. **No grain duplication:** Ensuring there are zero duplicate rows at our defined reporting grain (preventing data explosion in Power BI).
2. **No missing core metrics:** Verifying that our primary calculation fields (crime counts and population) contain no null values.

These automated quality checks utilise rigorous `assert` statements. If any unintended grain duplication or missing core metrics are detected, the pipeline will immediately halt and fail. Once validation successfully passes, the dataset is exported to a CSV ready for Power BI ingestion.

In [26]:
def export_and_validate(df_final, output_path):
    print("--- Starting Layer 5: Final Validation and Export ---")
    
    # 1. Final Grain Duplicate Check
    grain_cols = ['reported_by', 'month', 'lad_name', 'crime_type']
    duplicates = df_final.duplicated(subset=grain_cols).sum()
    print(f"Validation: Duplicate rows at reporting grain: {duplicates}")
    assert duplicates == 0, f"FAILED: {duplicates} unintended duplicate rows detected at the reporting grain!"
    print("Success: Dataset is perfectly aggregated. No many-to-many explosions.")
        
    # 2. Null Checks on critical reporting fields
    missing_counts = df_final[['crime_count', 'force_population']].isnull().sum()
    print("\nValidation: Missing values in core BI fields:")
    print(missing_counts)
    assert missing_counts['crime_count'] == 0, "FAILED: Missing values found in crime_count!"
    assert missing_counts['force_population'] == 0, "FAILED: Missing values found in force_population!" 
    
    # 3. Export to CSV
    print(f"\nExporting BI-ready dataset to: {output_path}")
    df_final.to_csv(output_path, index=False)
    print("Export complete. Pipeline successfully finished!")

# --- EXECUTION ---
FINAL_OUTPUT_PATH = './BI_Reporting_Dataset_Final.csv'

export_and_validate(df_bi_reporting, FINAL_OUTPUT_PATH)

--- Starting Layer 5: Final Validation and Export ---
Validation: Duplicate rows at reporting grain: 0
Success: Dataset is perfectly aggregated. No many-to-many explosions.

Validation: Missing values in core BI fields:
crime_count         0
force_population    0
dtype: int64

Exporting BI-ready dataset to: ./BI_Reporting_Dataset_Final.csv
Export complete. Pipeline successfully finished!
